# <font size="8">Image Segmentation: Embryo</font>


# Library

In [ ]:
#%pip install tifffile numpy matplotlib scikit-learn scipy pandas plotly
#%pip install ipython nbformat
#!pip install nbformat ipython ipykernel
#!pip install notebook nbconvert
#%pip install opencv-python 
#%pip install seaborn
#%pip install umap-learn   # For some reason, running install with % instead of ! is necessary to get the package to import correctly in this notebook. Don't ask me why, I have no idea
%pip install scanpy anndata
%pip install igraph leidenalg

# Importing standard libraries
import tifffile                                            # a library for reading and writing TIFF files, commonly used for handling large image data in scientific applications
import numpy as np             
import matplotlib.pyplot as plt                            # a plotting library used for creating static, interactive and animated visualisations in python. Uses BGR with imshow()
from sklearn.cluster import KMeans                         # implements the K-means clustering algorithm, an unsupervised learning method for grouping data points
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.colors as colors
from scipy import stats
from sklearn.decomposition import PCA
import pandas as pd
import plotly.express as px
import xml.etree.ElementTree as ET
import plotly
import plotly.express as px
import cv2 as cv # OpenCV library for image processing, mainly RGB 
import seaborn as sns
import umap

import plotly.io as pio
pio.renderers.default = 'vscode'


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Loading and Inspecting the Data (ome.tiff file)

In [ ]:
# Establishing ome.tif File Path
file_path = '/Users/tolgakiymak/Desktop/kings_masters/Extended_Research_Project/Data/Embryo_Data/MURINE_EMBRYO/MURINE_EMBRYO.ome.tif' 

# Loading the Raw Image Data
img_raw = tifffile.imread(file_path)

# Automatically Extract Channel Names from OME Metadata, if available
with tifffile.TiffFile(file_path) as tif:
    omexml_string = tif.ome_metadata

if omexml_string:
    namespaces = {'ome': 'http://www.openmicroscopy.org/Schemas/OME/2016-06'}
    root = ET.fromstring(omexml_string)
    channels = root.findall('.//ome:Channel', namespaces)
    all_channel_names = [c.attrib.get('Name', f"Channel_{i}") for i, c in enumerate(channels)]
else:
    print("Warning: No OME metadata found. Generating placeholder names.")
    all_channel_names = [f"Channel_{i}" for i in range(img_raw.shape[0])]

# Only drop channel 0 (0TIC) — always useless as it is the total ion count
# Se and any other poor channels will be dropped automatically by the quality assessment
img = np.delete(img_raw, [0], axis=0)
channel_names = [name for i, name in enumerate(all_channel_names) if i not in [0]]
num_channels = len(channel_names)

# Setup Colours Safely
base_colours = ['lime', 'blue', 'yellow', 'cyan', 'magenta', 'orange',
                'purple', 'pink', 'teal', 'lightgreen', 'gold', 'white']
qupath_colours = [base_colours[i % len(base_colours)] for i in range(num_channels)]

# Print Checks
print(f"Original Data Shape: {img_raw.shape}")
print(f"Data Shape after dropping 0TIC: {img.shape}")
print(f"Data Type: {img.dtype}")
print(f"Min value: {np.min(img)}")
print(f"Max value: {np.max(img)}\n")
print(f"Successfully loaded {num_channels} channels:")
print(channel_names)

# Loading and Inspecting Meta Data

In [ ]:
# Loading metadata CSV
metadata_path = '/Users/tolgakiymak/Desktop/kings_masters/Extended_Research_Project/Data/Embryo_Data/meta_data/meta_data.csv'
meta_df = pd.read_csv(metadata_path, index_col=0)

# Parse the min/max threshold lists from the metadata
import ast
import re
all_min_thresholds = ast.literal_eval(meta_df.loc['min thresholds', '0'])
all_max_thresholds = ast.literal_eval(meta_df.loc['max thresholds', '0'])
all_channel_names_meta = ast.literal_eval(meta_df.loc['processed isotopes', '0'])

# Build a lookup dict: channel name → (min, max)
# re.sub strips leading numbers from metadata names so '23Na' becomes 'Na', '56Fe' becomes 'Fe' etc.
# This makes them match channel_names_filtered which uses short names
threshold_lookup = {
    re.sub(r'^\d+', '', name): (all_min_thresholds[i], all_max_thresholds[i])
    for i, name in enumerate(all_channel_names_meta)
}

print("Threshold lookup built:")
for name, (mn, mx) in threshold_lookup.items():
    print(f"  {name}: min={mn}, max={mx}")

# Auto/Manually dropping poor channels

In [ ]:
# AUTOMATIC: drops channels flagged as POOR by the quality assessment
# Comment out this block if you want to use manual selection instead
#poor_channels = quality_df[quality_df['Flag'].str.startswith('POOR')]['Channel'].tolist()
#borderline_channels = quality_df[quality_df['Flag'].str.startswith('BORDERLINE')]['Channel'].tolist()
#print(f"POOR channels (will be dropped): {poor_channels}")
#print(f"BORDERLINE channels (review manually): {borderline_channels}")

# OPTION 2 — MANUAL: specify exactly which channels to drop
# Uncomment this block if you want full manual control
poor_channels = []  # start with empty list
manual_drop = ['Mn', 'K']  # ← add channel names here
poor_channels = poor_channels + manual_drop

# OPTIONAL: add extra channels to drop on top of automatic selection
# Only applies when using OPTION 1
#extra_drop = []  # add any extra channels here e.g. ['Se']
#poor_channels = poor_channels + extra_drop

# Filtering channels and image
good_channel_mask = [name not in poor_channels for name in channel_names]
channel_names_filtered = [name for name in channel_names if name not in poor_channels]
img_filtered = img[good_channel_mask]

print(f"\nOriginal channels: {len(channel_names)}")
print(f"Channels after dropping POOR: {len(channel_names_filtered)}")
print(f"Kept channels: {channel_names_filtered}")

# Masking the Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from skimage.morphology import (
    remove_small_objects, remove_small_holes,
    binary_erosion, binary_dilation, square,
)

# ===================================== #
#     FUNCTION FOR CALCULATING MASK     #
# ===================================== #
def threshold_SOR_fill_BED(
    image: np.ndarray,
    threshold: float,
    small_objects_removal: int = 0,
    fill_holes: bool = True,
    hole_area_threshold: int = int(1e4),
    BED_iterations: int = 0,
    connectivity: int = 2,
):
    # Threshold
    thresh_mask = image >= threshold
    thresh_image = image * thresh_mask

    # Small object removal
    if small_objects_removal > 0:
        SOR_mask = remove_small_objects(thresh_mask,
                                        min_size=small_objects_removal,
                                        connectivity=connectivity)
    else:
        SOR_mask = thresh_mask.copy()
    SOR_image = image * SOR_mask

    # Fill holes
    if fill_holes:
        fill_mask = remove_small_holes(SOR_mask,
                                       area_threshold=hole_area_threshold,
                                       connectivity=connectivity)
    else:
        fill_mask = SOR_mask.copy()
    fill_image = image * fill_mask

    # Binary erosion + dilation
    if BED_iterations > 0:
        BED_mask = fill_mask.copy()
        selem = square(3)
        for _ in range(BED_iterations):
            BED_mask = binary_erosion(BED_mask, selem)
            BED_mask = binary_dilation(BED_mask, selem)
    else:
        BED_mask = fill_mask.copy()
    BED_image = image * BED_mask

    return (
        thresh_image, thresh_mask.astype(np.uint8),
        SOR_image,    SOR_mask.astype(np.uint8),
        fill_image,   fill_mask.astype(np.uint8),
        BED_image,    BED_mask.astype(np.uint8),
    )


def percentile_95_excluding_zeros(image: np.ndarray) -> float:
    valid = np.isfinite(image) & (image > 0)
    if not np.any(valid):
        raise ValueError("No non-zero, finite pixels found in image.")
    return float(np.nanpercentile(image[valid], 95))


# ================== #
#       INPUTS       #
# ================== #
show_mask               = 1                              # 1 = show preview, 0 = skip
channel_for_mask        = 1                              # channel index used to build the mask (0=Na, 1=Mg, 2=P, 3=S, 4=Ca, 5=Fe, 6=Cu, 7=Zn, 8=Se)
mask_x_range            = [0, 1000]
mask_y_range            = [0, 1000]
mask_threshold          = 200
mask_small_object       = 800
mask_fill_holes         = False
mask_erosion_dilation   = 0


# ================== #
#     SHOW MASK      #
# ================== #
if show_mask > 0:
    mask_subregion = img[channel_for_mask]


    mask_preview = threshold_SOR_fill_BED(
        image=mask_subregion,
        threshold=mask_threshold,
        small_objects_removal=mask_small_object,
        fill_holes=mask_fill_holes,
        hole_area_threshold=10000,
        BED_iterations=mask_erosion_dilation,
    )

    mask_preview_thresh = percentile_95_excluding_zeros(mask_subregion)

    fig, (ax1, ax2, ax3) = plt.subplots(
        1, 3, sharex=True, sharey=True, figsize=(12, 4)
    )

    ax1.imshow(mask_subregion,   vmax=mask_preview_thresh, cmap='inferno_r', interpolation='nearest')
    ax2.imshow(mask_preview[6],  vmax=mask_preview_thresh, cmap='inferno_r', interpolation='nearest')
    ax3.imshow(mask_preview[7],                            cmap='gray',      interpolation='nearest')

    ax1.set_title("Original")
    ax2.set_title("Masked")
    ax3.set_title("Masked binary")

    for ax in (ax1, ax2, ax3):
        ax.set_xticks([])
        ax.set_yticks([])
        ax.axis("off")

    fig.suptitle(
        "Threshold = " + str(mask_threshold)
        + ", SOR = "   + str(mask_small_object)
        + ", Fill = "  + str(mask_fill_holes)
        + ", BED = "   + str(mask_erosion_dilation),
        fontsize=10,
    )
    plt.tight_layout(rect=[0, 0, 1, 0.9])
    plt.show()


# ================== #
#   Calculate MASK   #
# ================== #
mask_data = threshold_SOR_fill_BED(
    image=img[channel_for_mask],
    threshold=mask_threshold,
    small_objects_removal=mask_small_object,
    fill_holes=mask_fill_holes,
    hole_area_threshold=10000,
    BED_iterations=mask_erosion_dilation,
)[7]


# ================== #
#   Apply to df      #
# ================== #
n_channels, height, width = img_filtered.shape
n_pixels = height * width
img_reshaped = img_filtered.reshape(n_channels, n_pixels).T

tissue_indices = np.where(mask_data.flatten() > 0)[0]
tissue_indices_final = tissue_indices.copy()

df = pd.DataFrame(img_reshaped[tissue_indices], columns=channel_names_filtered)

print(f"Total pixels: {n_pixels:,}")
print(f"Tissue pixels in mask: {len(df):,}")
print(f"Pixels excluded: {n_pixels - len(df):,}")

# Setting Display Ranges from Metadata

In [ ]:
df.head()

In [ ]:
num_channels = img_filtered.shape[0]  # Use the actual dimension instead
print(f"img_filtered shape: {img_filtered.shape}")
print(f"num_channels value: {num_channels}")

In [ ]:
# I am storing the min/max for each channel in this list
channel_ranges = []

# This is the start of the for loop, going through all the channels
# "for i in...": This starts the loop. The first time it runs, i is 0. The next time, i is 1 and so on. i is the index counter
# "1" This is the number of rows (1)
# "num_channels_to_view": number of channels (5)
# "i + 1": The position of the current plot. Matplotlib counts positoons starting at 1, but python starts at 0. So when 1=0, we plot in position
for i in range(num_channels):
    
    # Directly fetching min and max from the metadata threshold lookup
    # If a channel name doesn't match, it will throw a KeyError so you can catch name mismatches early
    optimum_min, optimum_max = threshold_lookup[channel_names_filtered[i]]

    # Storing the result
    channel_ranges.append((optimum_min, optimum_max))

    print(f"Channel {i} ({channel_names_filtered[i]}): Min={optimum_min}, Max={int(optimum_max)}")

# Custom Colourmaps (1 row for quick visualisation)

In [ ]:
plt.figure(figsize=(20,5))

for i in range(num_channels):
  plt.subplot(1, num_channels, i + 1)

  vmin, vmax = channel_ranges[i]
  cmap_name = f"black_to_{qupath_colours[i]}"
  custom_cmap = LinearSegmentedColormap.from_list(cmap_name, ['black', qupath_colours[i]])

  plt.imshow(img_filtered[i], cmap=custom_cmap, vmin=vmin, vmax=vmax, origin='upper')
  plt.title(f"{channel_names_filtered[i]}\nColor: {qupath_colours[i]}", fontsize=8)
  plt.axis('off')

plt.tight_layout()
plt.show()

# Comparison between log and linear

In [ ]:
# Channel Settings
channels_to_test = len(channel_names_filtered)

# ← Save suggestions here so other cells can use them
scale_suggestions = {}

fig, axes = plt.subplots(nrows=channels_to_test, ncols=3, figsize=(18, 4 * channels_to_test))

for i in range(channels_to_test):
    
    channel_data = img_filtered[i]
    flat_data = channel_data.flatten()
    valid_data = flat_data[flat_data > 0]

    if len(valid_data) > 0:
        robust_max = np.percentile(valid_data, 99.9)
    else:
        robust_max = 1

    ax_lin = axes[i, 0]
    ax_lin.imshow(channel_data, cmap='inferno', vmax=robust_max, origin='upper')
    ax_lin.set_title(f"{channel_names_filtered[i]} - Linear Scale")
    ax_lin.axis('off')

    ax_log = axes[i, 1]
    # "vmin=1e-10" was changed to "vmin=1", stopping near-zero noise pixels from dragging the scale down
    # "vmax=robust_max" (99.9th percentile) was changed to "vmax=np.percentile(valid_data, 95)", bringing the top of the scale down so the colour range spreads more evenly across the tissue signal, rather than being dominated by a few very bright pixels
    ax_log.imshow(channel_data, cmap='magma',
                  norm=colors.LogNorm(vmin=1, vmax=np.percentile(valid_data, 95)), origin='upper')  
    ax_log.set_title(f"{channel_names_filtered[i]} - Log Scale")
    ax_log.axis('off')

    ax_hist = axes[i, 2]

    if len(valid_data) > 0:
        ax_hist.hist(valid_data, bins=100, color='gray', log=True)
        ax_hist.set_xlabel("Pixel Brightness")
        ax_hist.set_ylabel("Count (Log Scale)")

        skew = stats.skew(valid_data)
        cv = np.std(valid_data) / np.mean(valid_data)
        percentile_ratio = np.percentile(valid_data, 99) / (np.median(valid_data) + 1e-10)

        if skew > 3 and percentile_ratio > 50:
            suggestion = "LOG"
            suggest_color = 'red'
        elif skew > 2 or cv > 2:
            suggestion = "POSSIBLY LOG"
            suggest_color = 'orange'
        else:
            suggestion = "LINEAR"
            suggest_color = 'green'

        # ← Save suggestion for this channel
        scale_suggestions[channel_names_filtered[i]] = suggestion

        ax_hist.set_title(
            f"Distribution  |  Skew: {skew:.1f}  |  CV: {cv:.1f}  |  P99/Median: {percentile_ratio:.0f}x",
            fontsize=8
        )
        ax_hist.text(0.5, 0.9, f"Suggestion: {suggestion}", transform=ax_hist.transAxes,
                     color=suggest_color, weight='bold', ha='center')

plt.tight_layout()
plt.show()

print("\nScale suggestions saved:")
print(scale_suggestions)


#### OUTPUT EXPLAINED ####
# The histogram now shows the actual metric values (skew, CV, P00/median ratio) for every channel
# This means you are not just blindly trusting the suggestions, as you can see the numbers that drove the decision

# The code doesn't traces the tissue
# In the image, img_filtered[i] is a 2D array of shape (height, width) where each value is the elemental intensity measured at that physical location. 


# This cell produces three plots per channel: a linear scale image, a log scale image, and an intensity distribution histogram
# The histogram drives the automatic log/linear suggestion for each channel, saved into scale_suggestions for use in later cells

# LINEAR IMAGE (left column):
# Displays raw pixel intensities mapped directly to colour brightness
# vmax is set to the 99.9th percentile to prevent a few extreme hotspots from making the rest of the tissue appear black
# Uses inferno colormap

# LOG IMAGE (middle column):
# Displays the same data but with logarithmic colour scaling
# vmin=1 prevents near-zero noise pixels from dragging the colour scale down
# vmax=95th percentile brings the top of the scale down so colour spreads more evenly across tissue signal
# Uses magma colormap: black 
# Log scaling is useful when a channel has a few very bright hotspots that compress everything else into darkness

# HISTOGRAM (right column): three metrics drive the log/linear decision:
# Skew: measures how lopsided the distribution is toward bright outliers. Above 3 = heavily right-skewed, most pixels are dim but a few are extremely bright
# CV (Coefficient of Variation): standard deviation divided by mean, expressed as a ratio. Above 2.0 (equivalent to 200%) = signal is highly variable relative to its average
# CV of 200% = signal varies by twice its own mean. Highly variable, suggests either extreme heterogeneity or noise domination
# Percentile Ratio (P99/Median): 99th percentile divided by the median. Above 50x = the brightest 1% of pixels are more than 50 times brighter than the typical pixel

# DECISION LOGIC:
# LOG = skew above 3 AND percentile ratio above 50 (both conditions must be met)
# POSSIBLY LOG = skew above 2 OR CV above 2 (only one condition needed, moderate threshold)
# LINEAR = none of the above conditions met

# All suggestions are saved into scale_suggestions dictionary so downstream Plotly scatter plots
# can automatically display a second log-scaled plot for flagged channels

# <font size="8">UMAP</font>


# Implementing UMAP

In [ ]:
# This code cell is to get an idea of how many pixels/samples are now in the masked df
# In this case, we have 
df.shape[0]

In [ ]:
X_umap = umap.UMAP(
    n_components=3,
    n_neighbors=30,
    min_dist=0.0,             
    metric='cosine',
    random_state=42,
    verbose=True,
).fit_transform(df.values)

print(f"UMAP complete. Output shape: {X_umap.shape}")

In [ ]:
# Reuse the UMAP embedding you already computed
adata.obsm["X_umap"] = X_umap

# kNN graph in PCA space (the standard approach — clustering in full feature
# space via the kNN graph is more principled than clustering on the 2D UMAP)
sc.pp.neighbors(adata, n_neighbors=30, use_rep="X_pca")

# UMAP Visualisation 

In [ ]:
# Each pixel's three UMAP coordinates become its RGB values in the image
# Structurally similar pixels look colour-similar in their spatial image 

# Min-max normalise each UMAP axis to [0, 1] 
umap_norm = (X_umap - X_umap.min(axis=0)) / (X_umap.max(axis=0) - X_umap.min(axis=0))

# Build an empty (H, W, 3) image with white background
rgb_image = np.ones((height, width, 3), dtype=float)

# Drop tissue pixels in
ys, xs = np.unravel_index(tissue_indices_final, (height, width))
rgb_image[ys, xs, :] = umap_norm

plt.figure(figsize=(12, 7))
plt.imshow(rgb_image, origin='upper')
plt.title('UMAP RGB Embedding Map (3D UMAP RGB)')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objects as go

#   Building the per-pixel customdata for hover info                              
#   Shape: (height, width, 3 UMAP coords + 9 elements)        
channel_cols = channel_names          # the 9 elements
df_channels = df[channel_cols]
n_channels = len(channel_cols)

# float32 to keep payload size manageable (full-res images get heavy)
customdata = np.full((height, width, 3 + n_channels), np.nan, dtype=np.float32)

ys, xs = np.unravel_index(tissue_indices_final, (height, width))
customdata[ys, xs, 0:3] = X_umap.astype(np.float32)              # UMAP1, 2, 3
customdata[ys, xs, 3:]  = df_channels.values.astype(np.float32)  # 9 element intensities

#   Hover Template                                     
hover_lines = [
    'Pixel: (x=%{x}, y=%{y})',
    '──────────────',
    'UMAP1: %{customdata[0]:.2f}',
    'UMAP2: %{customdata[1]:.2f}',
    'UMAP3: %{customdata[2]:.2f}',
    '──────────────',
] + [
    f'{name}: %{{customdata[{i+3}]:.0f}}'
    for i, name in enumerate(channel_cols)
]
hovertemplate = '<br>'.join(hover_lines) + '<extra></extra>'


#   Display                                                  
fig = go.Figure(data=go.Image(
    z=(rgb_image * 255).astype(np.uint8),     # go.Image expects uint8
    customdata=customdata,
    hovertemplate=hovertemplate,
))

fig.update_layout(
    title='UMAP RGB Embedding Map — interactive hover',
    width=1400, height=780,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False, scaleanchor='x'),
    margin=dict(l=10, r=10, t=60, b=10),
    hoverlabel=dict(font_size=11, font_family='monospace'),
)
fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Subsample for plotly responsiveness
rng = np.random.default_rng(42)
n_plot = min(60_000, len(X_umap))
idx = rng.choice(len(X_umap), size=n_plot, replace=False)

# Each point's colour = its own UMAP-derived RGB
colours_rgb = umap_norm[idx]
colours_str = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})'
               for r, g, b in colours_rgb]

fig = go.Figure(data=go.Scatter3d(
    x=X_umap[idx, 0], y=X_umap[idx, 1], z=X_umap[idx, 2],
    mode='markers',
    marker=dict(size=2, color=colours_str, opacity=0.8),
    hovertemplate='UMAP1: %{x:.2f}<br>UMAP2: %{y:.2f}<br>UMAP3: %{z:.2f}<extra></extra>',
))

fig.update_layout(
    title='UMAP space — coloured by its own RGB encoding<br>'
          '<sub>This is the "legend" for your RGB embedding map</sub>',
    scene=dict(
        xaxis=dict(title='UMAP 1 → R', backgroundcolor='black'),
        yaxis=dict(title='UMAP 2 → G', backgroundcolor='black'),
        zaxis=dict(title='UMAP 3 → B', backgroundcolor='black'),
    ),
    width=900, height=800,
    margin=dict(l=0, r=0, t=80, b=0),
)
fig.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

channel_name = 'P'
suggestion = scale_suggestions.get(channel_name, 'LINEAR')

color_values_linear = df[channel_name].reset_index(drop=True)
color_values_log = np.log1p(color_values_linear)

# Always show linear
plt.figure(figsize=(8, 6))
plt.scatter(X_umap[:, 0], X_umap[:, 1],
            c=color_values_linear, s=0.1, cmap='inferno')
plt.colorbar(label=channel_name)
plt.xlabel('UMAP 1'); plt.ylabel('UMAP 2')
plt.title(f'UMAP 2D — {channel_name} (Linear)')
plt.show()

# Show log if flagged
if suggestion in ['LOG', 'POSSIBLY LOG']:
    plt.figure(figsize=(8, 6))
    plt.scatter(X_umap[:, 0], X_umap[:, 1],
                c=color_values_log, s=0.1, cmap='inferno')
    plt.colorbar(label=f'{channel_name} (log)')
    plt.xlabel('UMAP 1'); plt.ylabel('UMAP 2')
    plt.title(f'UMAP 2D — {channel_name} Log Scale (auto-flagged: {suggestion})')
    plt.show()

In [ ]:
channel_name = 'P'
suggestion = scale_suggestions.get(channel_name, 'LINEAR')

color_values_linear = df[channel_name].reset_index(drop=True)
color_values_log = np.log1p(color_values_linear)

# Always show linear
fig_linear = px.scatter_3d(
    x=X_umap[:, 0], y=X_umap[:, 1], z=X_umap[:, 2],
    color=color_values_linear,
    color_continuous_scale='inferno',
    title=f'UMAP 3D — {channel_name} (Linear)',
    labels={'x': 'UMAP 1', 'y': 'UMAP 2', 'z': 'UMAP 3'},
)
fig_linear.update_traces(marker=dict(size=2, opacity=0.6))
fig_linear.update_layout(width=900, height=800,
                         coloraxis_colorbar=dict(title=channel_name))
fig_linear.show()

# Show log if flagged
if suggestion in ['LOG', 'POSSIBLY LOG']:
    fig_log = px.scatter_3d(
        x=X_umap[:, 0], y=X_umap[:, 1], z=X_umap[:, 2],
        color=color_values_log,
        color_continuous_scale='inferno',
        title=f'UMAP 3D — {channel_name} Log Scale (auto-flagged: {suggestion})',
        labels={'x': 'UMAP 1', 'y': 'UMAP 2', 'z': 'UMAP 3'},
    )
    fig_log.update_traces(marker=dict(size=2, opacity=0.6))
    fig_log.update_layout(width=900, height=800,
                          coloraxis_colorbar=dict(title=f'{channel_name} (log)'))
    fig_log.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401

channel_name = 'P'
suggestion = scale_suggestions.get(channel_name, 'LINEAR')

color_values_linear = df[channel_name].reset_index(drop=True)
color_values_log = np.log1p(color_values_linear)

# Always show linear
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], X_umap[:, 2],
                c=color_values_linear, s=0.1, cmap='inferno')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2'); ax.set_zlabel('UMAP 3')
ax.set_title(f'UMAP 3D — {channel_name} (Linear)')
fig.colorbar(sc, ax=ax, shrink=0.7, label=channel_name)
plt.show()

# Show log if flagged
if suggestion in ['LOG', 'POSSIBLY LOG']:
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection='3d')
    sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], X_umap[:, 2],
                    c=color_values_log, s=0.1, cmap='inferno')
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2'); ax.set_zlabel('UMAP 3')
    ax.set_title(f'UMAP 3D — {channel_name} Log Scale (auto-flagged: {suggestion})')
    fig.colorbar(sc, ax=ax, shrink=0.7, label=f'{channel_name} (log)')
    plt.show()

In [ ]:
# Colouring by channel intensity, change index to explore other channels
color_channel_idx = 7
channel_name = channel_names_filtered[color_channel_idx]
suggestion = scale_suggestions.get(channel_name, 'LINEAR')

# All pixels are in df now 
color_values_linear = df.iloc[:, color_channel_idx].reset_index(drop=True)
color_values_log = np.log1p(color_values_linear)

print(f"{channel_name} suggestion: {suggestion}")

# Always show linear UMAP plot
fig_linear = px.scatter(
    x=X_umap[:, 0],
    y=X_umap[:, 1],
    color=color_values_linear,
    color_continuous_scale='hot',
    title=f'UMAP Plot — {channel_name} Linear Scale',
    labels={'x': 'UMAP 1', 'y': 'UMAP 2', 'z': 'UMAP 3'}
)
fig_linear.update_traces(marker=dict(size=5, opacity=0.6))
fig_linear.update_layout(
    coloraxis_colorbar=dict(title=channel_name),
    width=1000, height=800
)
fig_linear.show()

# Only show log UMAP plot if flagged as LOG or POSSIBLY LOG
if suggestion in ['LOG', 'POSSIBLY LOG']:
    fig_log = px.scatter(
        x=X_umap[:, 0],
        y=X_umap[:, 1],
        color=color_values_log,
        color_continuous_scale='hot',
        title=f'UMAP Plot — {channel_name} Log Scale (auto-flagged: {suggestion})',
        labels={'x': 'UMAP 1', 'y': 'UMAP 2', 'z': 'UMAP 3'}
    )
    fig_log.update_traces(marker=dict(size=5, opacity=0.6))
    fig_log.update_layout(
        coloraxis_colorbar=dict(title=f"{channel_name} (log)"),
        width=1000, height=800
    )
    fig_log.show()

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np

n = len(channel_names_filtered)
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
axes = axes.flatten()

for i, channel_name in enumerate(channel_names_filtered):

    # Raw intensity values for this channel — all pixels, matches X_umap row count
    color_vals = df.iloc[:, i].reset_index(drop=True)

    suggestion = scale_suggestions.get(channel_name, 'LINEAR')
    if suggestion in ['LOG', 'POSSIBLY LOG']:
        color_vals = np.log1p(color_vals)
        scale_label = '(log)'
    else:
        scale_label = ''

    sc = axes[i].scatter(
        X_umap[:, 0], 
        X_umap[:, 1], 
        c=color_vals,
        cmap='hot',
        s=0.3,
        rasterized=True
    )
    plt.colorbar(sc, ax=axes[i], shrink=0.7)
    axes[i].set_title(f'{channel_name} {scale_label}', fontsize=12)
    axes[i].axis('off')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('UMAP — All Elements', fontsize=18, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

X = df.values.astype(float)
X_norm = np.clip(X / np.percentile(X, 99, axis=0), 0, 1)
dominant = X_norm.argmax(axis=1)

n = df.shape[1]                              # <-- use df, not the name list
labels = list(df.columns)                    # <-- and use df's column names
palette = plt.cm.tab10(np.linspace(0, 1, n)) if n <= 10 else plt.cm.tab20(np.linspace(0, 1, n))
pixel_colours = palette[dominant]

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(X_umap[:, 0], 
           X_umap[:, 1], 
           c=pixel_colours, s=0.3, 
           alpha=0.7)

handles = [mpatches.Patch(color=palette[i], label=labels[i]) for i in range(n)]
ax.legend(handles=handles, title='Dominant element',
          bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)

ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.set_title('UMAP coloured by dominant element per pixel')
plt.tight_layout()
plt.show()

# <font size="8">K-means</font>


# Elbow Method

In [ ]:
%pip install kneed
from kneed import KneeLocator

In [ ]:
# Finding the optimal 'K' (Elbow Method)
# WCSS (Within-Cluster Sum of Squares) measures how tight the clusters are
# Lower WCSS = pixels within each cluster are more similar to each other
# We test different values of k to find where adding more clusters stops being useful


# Testing 1 through 10 clusters
# We run KMeans on X_pca
# Using all 12 PCs:  KMeans clusters on both meaningful signal and noise
# Using just PC1-PC4:  KMeans clusters only on the variation that actually matters
X_kmeans = X_umap

wcss = []
max_k = 10

for i in range(1, max_k + 1):
    km = KMeans(n_clusters=i, init='k-means++', max_iter=300,
                n_init='auto', random_state=42)
    km.fit(X_kmeans)
    wcss.append(km.inertia_)

# Auto-detect elbow
kl = KneeLocator(range(1, max_k + 1), wcss,
                 curve='convex', direction='decreasing')
optimal_k = kl.elbow
print(f"KneeLocator suggests k = {optimal_k}")

plt.figure(figsize=(10, 5))
plt.plot(range(1, max_k + 1), wcss, marker='o', linestyle='--', color='blue')
if optimal_k is not None:
    plt.axvline(optimal_k, color='red', linestyle=':',
                label=f'Elbow (k = {optimal_k})')
    plt.legend()
plt.title("Elbow Method (KneeLocator) — UMAP 3D")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("WCSS")
plt.xticks(range(1, max_k + 1))
plt.grid(True, alpha=0.5)
plt.show()

# OUTPUT EXPLAINED:
# Look for the elbow — the point where the line stops dropping steeply and flattens out
# That elbow point is your optimal k value
# The elbow is often around k=3-5
# representing broad tissue compartments (e.g. dense tissue, sparse tissue, background)

# Applying K-means

In [ ]:
# Applying K-means

# Use kneedle's choice, or override manually if needed
chosen_k = optimal_k if optimal_k is not None else 4

kmeans = KMeans(n_clusters=chosen_k, init='k-means++', max_iter=300,
                n_init='auto', random_state=42)
cluster_labels = kmeans.fit_predict(X_umap)
df['cluster'] = cluster_labels.astype(str)

print(f"K-Means complete using k={chosen_k} on UMAP coordinates.")
print("\n--- Cluster Distribution ---")
counts = df['cluster'].value_counts().sort_index()
for cid, count in counts.items():
    pct = (count / len(df)) * 100
    print(f"Cluster {cid}: {count:,} pixels ({pct:.1f}%)")

In [ ]:
# Leiden clustering at multiple resolutions
# Lower resolution = fewer, broader clusters; higher = more, finer clusters.
# This is the principled alternative to manually picking k or min_cluster_size.

for res in [0.1, 0.3, 0.5, 1.0, 2.0]:
    sc.tl.leiden(
        adata,
        resolution=res,
        key_added=f"leiden_res_{res:.2f}",
        flavor="igraph",
        n_iterations=2,
        directed=False,
    )

# Visualise the resolution sweep on UMAP
sc.pl.umap(
    adata,
    color=[f"leiden_res_{r:.2f}" for r in [0.1, 0.3, 0.5, 1.0, 2.0]],
    ncols=3,
    legend_loc="on data",
)

In [ ]:
# Pick whichever Leiden resolution you settled on
chosen_resolution = "leiden_res_0.50"
adata.obs["leiden"] = adata.obs[chosen_resolution]

# Rank elements per cluster (Wilcoxon, same test scanpy uses for marker genes)
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")

# Dotplot: rows = clusters, columns = elements, dot size = fraction expressing,
# colour = mean expression. For elemental data this reads as enrichment per cluster.
sc.pl.rank_genes_groups_dotplot(
    adata,
    groupby="leiden",
    n_genes=len(channel_names_filtered),  # show all 9 elements
    standard_scale="var",
)

# Violin plot of each element across clusters
sc.pl.stacked_violin(
    adata,
    var_names=channel_names_filtered,
    groupby="leiden",
    standard_scale="var",
)

# K-means Clustering on UMAP

In [ ]:
fig = px.scatter(
    x=X_umap[:, 0],
    y=X_umap[:, 1],
    color=cluster_labels.astype(str),
    title='K-Means Clusters on UMAP Embedding',
    labels={'x': 'UMAP 1', 'y': 'UMAP 2', 'color': 'K-Means Cluster'}
)
fig.update_traces(marker=dict(size=5, opacity=0.6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(width=1000, height=800)
fig.show()

# Mapping Clusters to the Original Image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Preparing the 2D grid
# height and width should already be in the namespace from the filtering step
cluster_map = np.full((height, width), fill_value=-1, dtype=int)

# Mapping the labels back to their spatial positions
# Since cluster_labels and tissue_indices_final are the same length, we map directly
cluster_map.flat[tissue_indices_final] = cluster_labels.astype(int)

# Setup the custom Colormap
n_clusters = len(np.unique(cluster_labels))
print(f"Mapping {n_clusters} clusters back to {height}x{width} spatial grid.")

# Colors for the legend and map
cluster_colours = ['steelblue', 'red', 'limegreen', 'orange', 'purple', 'cyan']
cmap_colours = ['white'] + cluster_colours[:n_clusters]
cmap = ListedColormap(cmap_colours)

# Create Legend
legend_elements = [Patch(facecolor='white', edgecolor='black', label='Background (Masked)')]
for i in range(n_clusters):
    legend_elements.append(Patch(facecolor=cluster_colours[i], label=f'Cluster {i}'))

# Generating Side-by-Side Plots
# Note: This will loop through ALL filtered channels. 
# If you have many, you might want to slice channel_names_filtered[:3] to test.
for i, channel_name in enumerate(channel_names):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Plot Cluster Map
    im0 = axes[0].imshow(cluster_map, cmap=cmap, vmin=-1, vmax=n_clusters - 1, origin='upper')
    axes[0].set_title(f'K-Means Cluster Map', fontsize=14)
    axes[0].axis('off')
    axes[0].legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.3, 1))

    # Plot Raw Isotope Channel for Comparison
    # Using threshold_lookup to keep brightness consistent with previous steps
    vmin_ch, vmax_ch = threshold_lookup.get(channel_name, (0, np.percentile(img[i], 99)))
    
    axes[1].imshow(img_filtered[i], cmap='inferno', vmin=vmin_ch, vmax=vmax_ch, origin='upper')
    axes[1].set_title(f'Reference: {channel_name}', fontsize=14)
    axes[1].axis('off')

    plt.suptitle(f'Cluster Map vs {channel_name}', fontsize=16, y=1.05)
    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.cluster import KMeans
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import numpy as np
import matplotlib.pyplot as plt

# Configuration
# Match n_tiers to the number of clusters used in your main K-Means
n_tiers = 3 

# Syncing these exactly with your previous "cluster_colours" list
tier_colours = ['steelblue', 'red', 'limegreen', 'orange', 'purple', 'cyan']
cmap_colours = ['white'] + tier_colours[:n_tiers]
cmap_tiers = ListedColormap(cmap_colours)

# Create Legend with synced colors and descriptive labels
legend_elements = [Patch(facecolor='white', edgecolor='black', label='Background (Masked)')]
tier_labels = ["Cluster 0", "Cluster 1", "Cluster 2"] 

for k in range(n_tiers):
    label = tier_labels[k] if k < len(tier_labels) else f"Tier {k}"
    legend_elements.append(Patch(facecolor=tier_colours[k], label=label))

# Iterate through each channel
for i, channel_name in enumerate(channel_names_filtered):

    # Pull the specific channel data from our dataframe
    channel_values = df[channel_name].values.reshape(-1, 1)

    # Run K-Means for this specific channel
    kmeans_tier = KMeans(n_clusters=n_tiers, init='k-means++', max_iter=300, n_init='auto', random_state=42)
    raw_labels = kmeans_tier.fit_predict(channel_values)

    # SORTING: Ensure Cluster 0 is the lowest mean intensity
    cluster_means = [channel_values[raw_labels == k].mean() for k in range(n_tiers)]
    sorted_indices = np.argsort(cluster_means)
    
    mapping = {old_label: new_label for new_label, old_label in enumerate(sorted_indices)}
    sorted_labels = np.array([mapping[label] for label in raw_labels])

    # Spatial Mapping
    tier_map = np.full((height, width), fill_value=-1, dtype=int)
    tier_map.flat[tissue_indices_final] = sorted_labels

    # Visualisation
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Tiered Cluster Map
    axes[0].imshow(tier_map, cmap=cmap_tiers, vmin=-1, vmax=n_tiers-1, origin='upper')
    axes[0].set_title(f'{channel_name}: K-means Clusters Map', fontsize=14)
    axes[0].axis('off')
    axes[0].legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.25, 1))

    # Raw Reference
    vmin_ch, vmax_ch = threshold_lookup.get(channel_name, (0, np.percentile(img_filtered[i], 99)))
    axes[1].imshow(img_filtered[i], cmap='inferno', vmin=vmin_ch, vmax=vmax_ch, origin='upper')
    axes[1].set_title(f'Raw Channel: {channel_name}', fontsize=14)
    axes[1].axis('off')

    plt.suptitle(f'Cluster Map Vs {channel_name}', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

# <font size="8">HDBscan</font>


# Stands for "Hierarchical Density-Based Spatial Clustering of Applications with Noise"
# - Better than K-means at handling outliers/noise (K-means forces every point into a cluster, often undesirable)

In [ ]:
#%pip install hdbscan

import hdbscan
import sklearn.cluster as cluster 
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score

In [ ]:
results = []
for mcs in [50, 100, 200, 300, 500, 800, 1500]:
    for ms in [5, 25, 50]:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms,
            cluster_selection_method='eom',
            core_dist_n_jobs=-1,
        )
        labels = clusterer.fit_predict(X_umap)
        n_clusters = labels.max() + 1
        n_noise = (labels == -1).sum()
        # Crude balance metric: how dominant is the biggest cluster?
        if n_clusters > 0:
            sizes = pd.Series(labels[labels >= 0]).value_counts()
            biggest_pct = sizes.iloc[0] / sizes.sum() * 100
        else:
            biggest_pct = 100
        results.append({
            'min_cluster_size': mcs, 'min_samples': ms,
            'n_clusters': n_clusters,
            'noise_pct': 100 * n_noise / len(labels),
            'biggest_cluster_pct': biggest_pct,
        })

pd.DataFrame(results).sort_values('biggest_cluster_pct')

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=200,
    min_samples=25,
    cluster_selection_method='leaf',     # ← was 'eom'; 'leaf' splits density peaks aggressively
    core_dist_n_jobs=-1,
)
hdb_labels = clusterer.fit_predict(X_umap)

n_hdb = hdb_labels.max() + 1
n_noise = (hdb_labels == -1).sum()
print(f"Clusters found: {n_hdb}")
print(f"Noise pixels:   {n_noise:,} ({100 * n_noise / len(hdb_labels):.2f}%)")

# Cluster size distribution
import pandas as pd
sizes = pd.Series(hdb_labels[hdb_labels >= 0]).value_counts().sort_index()
print("\nCluster sizes:")
for cid, count in sizes.items():
    pct = count / sizes.sum() * 100
    print(f"  Cluster {cid}: {count:>7,} pixels ({pct:5.2f}%)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Generate enough distinct colours for whatever leaf produced
if n_hdb <= 10:
    palette = plt.cm.tab10(np.linspace(0, 1, n_hdb))
elif n_hdb <= 20:
    palette = plt.cm.tab20(np.linspace(0, 1, n_hdb))
else:
    palette = plt.cm.gist_ncar(np.linspace(0, 1, n_hdb))

# Find the dominant cluster and desaturate its colour to light grey
biggest_cluster = sizes.idxmax()
hdb_colours = [tuple(c) for c in palette]
hdb_colours[biggest_cluster] = (0.91, 0.91, 0.91, 1.0)   # light grey for bulk tissue

cmap_colours = ['white', '#BDBDBD'] + hdb_colours       # white = mask, grey = noise
cmap_hdb = ListedColormap(cmap_colours)

# Build the spatial map
hdb_map = np.full((height, width), fill_value=-2, dtype=int)
hdb_map.flat[tissue_indices_final] = hdb_labels

legend_elements = [
    Patch(facecolor='white', edgecolor='black', label='Background (Masked)'),
    Patch(facecolor='#BDBDBD', label='HDBSCAN Noise'),
]
for k in range(n_hdb):
    label = f'Cluster {k} (bulk tissue)' if k == biggest_cluster else f'Cluster {k}'
    legend_elements.append(Patch(facecolor=hdb_colours[k], label=label))

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(hdb_map, cmap=cmap_hdb, vmin=-2, vmax=n_hdb - 1, origin='upper')
ax.set_title(f"HDBSCAN Cluster Map — 'leaf' selection "
             f"(k={n_hdb}, {n_noise:,} noise pixels)", fontsize=14)
ax.axis('off')
ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.35, 1),
          frameon=True, framealpha=0.9, edgecolor='lightgrey', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
import hdbscan
import pandas as pd

results = []
for mcs in [100, 200, 300, 500, 800]:
    for ms in [5, 25, 50]:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms,
            cluster_selection_method='leaf',
            core_dist_n_jobs=-1,
        )
        labels = clusterer.fit_predict(X_umap)
        n_clusters = labels.max() + 1
        n_noise = (labels == -1).sum()
        if n_clusters > 0:
            cluster_sizes = pd.Series(labels[labels >= 0]).value_counts()
            biggest_pct = cluster_sizes.iloc[0] / cluster_sizes.sum() * 100
        else:
            biggest_pct = 100
        results.append({
            'min_cluster_size': mcs,
            'min_samples': ms,
            'n_clusters': n_clusters,
            'noise_pct': round(100 * n_noise / len(labels), 3),
            'biggest_cluster_pct': round(biggest_pct, 2),
        })

sweep_df = pd.DataFrame(results).sort_values('biggest_cluster_pct')
sweep_df

# What you're looking for in the sorted table:
#   biggest_cluster_pct < 60%  : balanced clustering
#   noise_pct < 5%             : most pixels confidently assigned
#   n_clusters between 5 and 20: biologically interpretable

In [ ]:
import hdbscan
from sklearn.preprocessing import StandardScaler

# Standardise so each channel contributes equally to euclidean distance
# (Different from your UMAP path which uses cosine; this is just a fairness check.)
X_raw = StandardScaler().fit_transform(df.values)

clusterer_raw = hdbscan.HDBSCAN(
    min_cluster_size=500,
    min_samples=50,
    cluster_selection_method='leaf',
    metric='euclidean',
    core_dist_n_jobs=-1,
)
hdb_labels_raw = clusterer_raw.fit_predict(X_raw)

n_raw = hdb_labels_raw.max() + 1
n_noise_raw = (hdb_labels_raw == -1).sum()
print(f"HDBSCAN on raw 9D channels:")
print(f"  Clusters: {n_raw}")
print(f"  Noise:    {n_noise_raw:,} ({100 * n_noise_raw / len(hdb_labels_raw):.2f}%)")

# If this gives broadly similar k to UMAP-based HDBSCAN → UMAP isn't inventing structure
# If wildly different → UMAP is shaping the clustering significantly

In [ ]:
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score

# Compare only on pixels HDBSCAN didn't label as noise
mask = hdb_labels != -1
ari = adjusted_rand_score(cluster_labels[mask], hdb_labels[mask])
ami = adjusted_mutual_info_score(cluster_labels[mask], hdb_labels[mask])

print("K-means vs HDBSCAN agreement (excluding HDBSCAN noise pixels)")
print("=" * 60)
print(f"  Adjusted Rand Index (ARI):         {ari:.3f}")
print(f"  Adjusted Mutual Information (AMI): {ami:.3f}")
print()
print("Interpretation:")
print("  1.0  = identical partitioning")
print("  0.5+ = strong agreement, both methods see same biology")
print("  0.2  = weak agreement, methods disagree on boundaries")
print("  0.0  = random / no agreement")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# K-means side
kmeans_colours = ['#4C72B0', '#DD8452', '#55A868', '#C44E52',
                  '#8172B2', '#CCB974', '#64B5CD', '#937860']
n_km = len(np.unique(cluster_labels))
km_cmap = ListedColormap(['white'] + kmeans_colours[:n_km])

km_map = np.full((height, width), fill_value=-1, dtype=int)
km_map.flat[tissue_indices_final] = cluster_labels.astype(int)

# HDBSCAN side (re-using hdb_map and cmap_hdb from Cell B)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

axes[0].imshow(km_map, cmap=km_cmap, vmin=-1, vmax=n_km - 1, origin='upper')
axes[0].set_title(f'K-means (k={n_km})\nForces every pixel into a cluster',
                  fontsize=13)
axes[0].axis('off')

axes[1].imshow(hdb_map, cmap=cmap_hdb, vmin=-2, vmax=n_hdb - 1, origin='upper')
axes[1].set_title(f"HDBSCAN-leaf (k={n_hdb})\n"
                  f"Identifies density peaks; {n_noise:,} pixels labelled noise",
                  fontsize=13)
axes[1].axis('off')

fig.suptitle('K-means vs HDBSCAN — same UMAP, different questions', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

n_hdb = hdb_labels.max() + 1
n_noise = (hdb_labels == -1).sum()

# Generate enough distinct colours for whatever leaf produced
if n_hdb <= 10:
    palette = plt.cm.tab10(np.linspace(0, 1, n_hdb))
elif n_hdb <= 20:
    palette = plt.cm.tab20(np.linspace(0, 1, n_hdb))
else:
    palette = plt.cm.gist_ncar(np.linspace(0, 1, n_hdb))

# Find the dominant cluster and desaturate its colour to light grey
sizes = pd.Series(hdb_labels[hdb_labels >= 0]).value_counts()
biggest_cluster = sizes.idxmax()
hdb_colours = [tuple(c) for c in palette]
hdb_colours[biggest_cluster] = (0.91, 0.91, 0.91, 1.0)   # bulk tissue → light grey

cmap_colours = ['white', '#BDBDBD'] + hdb_colours        # white = mask, grey = noise
cmap_hdb = ListedColormap(cmap_colours)

# Build spatial map: -2 = background, -1 = noise, 0..n-1 = clusters
hdb_map = np.full((height, width), fill_value=-2, dtype=int)
hdb_map.flat[tissue_indices_final] = hdb_labels

legend_elements = [
    Patch(facecolor='white', edgecolor='black', label='Background (Masked)'),
    Patch(facecolor='#BDBDBD', label='HDBSCAN Noise'),
]
for k in range(n_hdb):
    label = f'Cluster {k} (bulk tissue)' if k == biggest_cluster else f'Cluster {k}'
    legend_elements.append(Patch(facecolor=hdb_colours[k], label=label))

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(hdb_map, cmap=cmap_hdb, vmin=-2, vmax=n_hdb - 1, origin='upper',
          interpolation='nearest')
ax.set_title(f"HDBSCAN Cluster Map — 'leaf' selection "
             f"(k={n_hdb}, {n_noise:,} noise pixels)", fontsize=14)
ax.axis('off')
ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.35, 1),
          frameon=True, framealpha=0.9, edgecolor='lightgrey', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# --- Convert RGBA tuples from the matplotlib palette to hex strings ---
def rgba_to_hex(rgba):
    r, g, b = [int(c * 255) for c in rgba[:3]]
    return f'#{r:02x}{g:02x}{b:02x}'

# Order: background (-2), noise (-1), clusters 0..n_hdb-1
hex_colours = ['#FFFFFF', '#BDBDBD'] + [rgba_to_hex(c) for c in hdb_colours]

# --- Build a stepped (discrete) colorscale for plotly ---
# Plotly Heatmap needs a continuous-looking colorscale, so we duplicate boundaries
# to force hard stops between cluster colours.
n_steps = n_hdb + 2
boundaries = np.linspace(0, 1, n_steps + 1)
colorscale = []
for i, c in enumerate(hex_colours):
    colorscale.append([boundaries[i], c])
    colorscale.append([boundaries[i + 1], c])

# --- Friendly cluster names for hover ---
def cluster_name(k):
    if k == -2: return 'Background'
    if k == -1: return 'Noise'
    if k == biggest_cluster: return f'Cluster {k} (bulk tissue)'
    return f'Cluster {k}'

# Build a string array same shape as hdb_map so hovertemplate can show names
hover_text = np.vectorize(cluster_name)(hdb_map)

# --- Plot ---
fig = go.Figure(data=go.Heatmap(
    z=hdb_map,
    customdata=hover_text,
    colorscale=colorscale,
    zmin=-2, zmax=n_hdb - 1,
    showscale=False,
    hovertemplate='x: %{x}<br>y: %{y}<br>%{customdata}<extra></extra>',
))

fig.update_layout(
    title=dict(
        text=f"HDBSCAN Cluster Map — interactive  "
             f"(k={n_hdb}, {n_noise:,} noise pixels)",
        x=0.5, xanchor='center',
    ),
    width=1300, height=750,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False, scaleanchor='x', autorange='reversed'),
    plot_bgcolor='white',
    margin=dict(l=20, r=20, t=60, b=20),
)
fig.show()